实战项目：Workflow vs Agent 对比
方案一：Workflow 实现

In [ ]:
# 导入正则表达式和类型定义模块
import re
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# === State 定义（内容审核的状态数据结构） ===
class ContentModerationState(TypedDict):
    content: str                # 待审核的内容
    has_profanity: bool         # 是否包含脏话
    has_spam_pattern: bool      # 是否包含垃圾信息模式
    has_sensitive_topics: bool  # 是否包含敏感话题
    decision: str               # 审核决策: "approved" | "rejected" | "needs_review"
    reason: str                 # 审核原因说明

# === 节点实现（每个节点负责一项检查） ===

def check_profanity(state: ContentModerationState) -> dict:
    """检查脏话（规则基于）"""
    content = state["content"].lower()
    # 脏话关键词列表（实际应用中从数据库加载）
    profanity_list = ["脏话1", "脏话2", "敏感词"]
    has_profanity = any(word in content for word in profanity_list)
    return {"has_profanity": has_profanity}

def check_spam(state: ContentModerationState) -> dict:
    """检查垃圾信息模式"""
    content = state["content"]

    # 规则1: 重复字符检测（连续重复5次以上）
    has_repeat = bool(re.search(r'(.)\1{5,}', content))

    # 规则2: 过多链接检测
    has_many_links = content.count("http") > 3

    # 规则3: 全大写检测
    has_all_caps = content.isupper() and len(content) > 20

    # 满足任一规则即判定为垃圾信息
    has_spam = has_repeat or has_many_links or has_all_caps
    return {"has_spam_pattern": has_spam}

def check_sensitive_topics(state: ContentModerationState) -> dict:
    """检查敏感话题"""
    content = state["content"].lower()
    # 敏感关键词列表（简化示例）
    sensitive_keywords = ["政治", "暴力", "色情"]

    has_sensitive = any(keyword in content for keyword in sensitive_keywords)
    return {"has_sensitive_topics": has_sensitive}

def make_decision(state: ContentModerationState) -> dict:
    """综合决策节点：根据所有检查结果做出最终判定"""
    has_profanity = state.get("has_profanity", False)
    has_spam = state.get("has_spam_pattern", False)
    has_sensitive = state.get("has_sensitive_topics", False)

    has_sensitive = state.get("has_sensitive_topics", False)

    # 按优先级做出决策
    if has_profanity:
        return {
            "decision": "rejected",
            "reason": "包含不当语言"
        }
    elif has_sensitive:
        return {
            "decision": "needs_review",
            "reason": "包含敏感话题，需人工审核"
        }
    elif has_spam:
        return {
            "decision": "rejected",
            "reason": "疑似垃圾信息"
        }
    else:
        return {
            "decision": "approved",
            "reason": "内容正常"
        }

    # === 构建 Workflow（固定流程的工作流） ===
workflow_graph = StateGraph(ContentModerationState)

# 添加所有检查节点和决策节点
workflow_graph.add_node("check_profanity", check_profanity)
workflow_graph.add_node("check_spam", check_spam)
workflow_graph.add_node("check_sensitive", check_sensitive_topics)
workflow_graph.add_node("decide", make_decision)

# 固定的执行顺序：脏话检查 -> 垃圾信息检查 -> 敏感话题检查 -> 决策
workflow_graph.add_edge(START, "check_profanity")
workflow_graph.add_edge("check_profanity", "check_spam")
workflow_graph.add_edge("check_spam", "check_sensitive")
workflow_graph.add_edge("check_sensitive", "decide")
workflow_graph.add_edge("decide", END)

# 编译工作流
workflow_app = workflow_graph.compile()

In [ ]:
# 可视化工作流图结构
from IPython.display import Image, display

try:
    display(Image(workflow_app.get_graph(xray=True).draw_png()))
except Exception as e:
    print(f"Graphviz 渲染失败: {e}")
    print("\n使用 Mermaid 文本方式显示:")
    print(workflow_app.get_graph(xray=True).draw_mermaid())

In [ ]:
# === 测试 Workflow 方案 ===
# 准备不同类型的测试内容
test_contents = [
    "这是一条正常的评论。",        # 正常内容
    "这包含脏话1的内容",           # 包含脏话
    "AAAAAAAAAAAAAAAAAAAA买买买！！！。",  # 垃圾信息（重复字符）
    "讨论政治话题的内容"            # 敏感话题
]

print(f"=== Workflow 实现结果 ===\n")
# 逐个测试并打印结果
for content in test_contents:
    result = workflow_app.invoke({"content": content})
    print(f"内容: {content}")
    print(f"决策: {result['decision']}")
    print(f"原因: {result['reason']}\n")

方案二：Agent 实现

In [ ]:
# 导入 Agent 方案所需的模块
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model        # 聊天模型初始化工具
from langchain.messages import SystemMessage, HumanMessage  # 消息类型定义
from dotenv import load_dotenv  # 环境变量加载
import os

In [ ]:
# === State 定义（Agent 方案的状态） ===
class AgentModerationState(TypedDict):
    content: str        # 待审核内容
    analysis: str       # LLM 分析结果
    decision: str       # 审核决策: "approved" | "rejected" | "needs_review"
    reason: str         # 决策原因
    confidence: float   # 置信度分数

# 加载环境变量（从 .env 文件读取 API 密钥）
load_dotenv()
API_KEY = os.getenv('OPENAI_API_KEY')

# === LLM 初始化（创建语言模型实例） ===
model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    base_url="https://api.openai.com/v1",
    api_key=API_KEY,
    temperature=0.0  # 设为0确保输出确定性
)

# === 节点实现 ===
def analyze_content(state: AgentModerationState) -> dict:
    """使用 LLM 分析内容（Agent 的核心：让 AI 做判断）"""
    content = state["content"]

    # 系统提示词：定义 LLM 的分析任务和输出格式
    system_prompt = """你是一个内容审核助手。分析给定的内容，判断是否包含：
    1. 不当语言（脏话、侮辱）
    2. 垃圾信息（广告、刷屏）
    3. 敏感话题（政治、暴力、色情）

    请以 JSON 格式返回：
    {
        "has_issues": true/false,
        "issues": ["issue1", "issue2"],
        "severity": "low/medium/high",
        "confidence": 0.0-1.0
    }
    """

    # 构建消息列表并调用 LLM
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"内容：{content}")
    ]

    response = model.invoke(messages)
    return {"analysis": response.content}

def make_agent_decision(state: AgentModerationState) -> dict:
    """基于 LLM 分析结果做最终决策"""
    analysis = state["analysis"]
    content = state["content"]

    # 决策提示词
    system_prompt = """基于之前的分析结果，做出审核决策：
    - approved:   内容正常，通过
    - rejected:   明显违规，直接拒绝
    - needs_review: 需要人工审核

    返回 JSON 格式:
    {
        "decision": "approved/rejected/needs_review",
        "reason": "简短说明",
        "confidence": 0.0-1.0
    }
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"原内容: {content}\n\n分析结果: {analysis}")
    ]

    response = model.invoke(messages)

    # 解析 LLM 的 JSON 响应（实际应用中应该用 JSON mode）
    import json
    try:
        result = json.loads(response.content)
        return {
            "decision": result["decision"],
            "reason": result["reason"],
            "confidence": result["confidence"]
        }
    except:
        # 解析失败时的兜底处理
        return {
            "decision": "needs_review",
            "reason": "解析失败，需人工审核",
            "confidence": 1.0
        }

# === 路由函数（条件分支：决定走自动决策还是人工审核） ===
def should_auto_decide(state: AgentModerationState) -> Literal["decide", "review"]:
    """根据分析结果决定是否需要人工"""
    # 如果分析中提到高风险，直接转人工审核
    if "high" in state.get("analysis", "").lower():
        return "review"
    return "decide"

def human_review_placeholder(state: AgentModerationState) -> dict:
    """人工审核占位符（实际应用中可使用 interrupt 中断等待人工操作）"""
    return {
        "decision": "needs_review",
        "reason": "已转人工审核",
        "confidence": 1.0
    }

# === 构建 Agent Graph ===
agent_graph = StateGraph(AgentModerationState)

# 添加节点
agent_graph.add_node("analyze", analyze_content)         # LLM 分析节点
agent_graph.add_node("decide", make_agent_decision)       # 自动决策节点
agent_graph.add_node("human_review", human_review_placeholder)  # 人工审核节点

# 定义边和条件路由
agent_graph.add_edge(START, "analyze")
agent_graph.add_conditional_edges(
    "analyze",
    should_auto_decide,
    {
        "decide": "decide",        # 低/中风险 -> 自动决策
        "review": "human_review"   # 高风险 -> 人工审核
    }
)

agent_graph.add_edge("decide", END)
agent_graph.add_edge("human_review", END)

# 编译 Agent 图
agent_app = agent_graph.compile()

# 可视化 Agent 图结构
from IPython.display import Image, display
try:
    display(Image(agent_app.get_graph(xray=True).draw_png()))
except Exception as e:
    print(f"Graphviz 渲染失败: {e}")
    print("\n使用 Mermaid 文本方式显示:")
    print(agent_app.get_graph(xray=True).draw_mermaid())

In [ ]:
# === 测试 Agent 方案 ===
print("\n=== Agent 实现结果 ===\n")
for content in test_contents:
    # 调用 Agent 图进行审核
    result = agent_app.invoke({"content": content})
    print(f"内容: {content}")
    print(f"分析: {result.get('analysis', 'N/A')[:100]}...")  # 只显示前100字符
    print(f"决策: {result['decision']}")
    print(f"原因: {result['reason']}")
    print(f"置信度: {result.get('confidence', 'N/A')}\n")

混合方案（推荐）

In [ ]:
# 混合方案：结合 Workflow 和 Agent 的优势
# 第一层：Workflow 快速过滤明显违规（速度快、成本低）
# 第二层：Agent 处理边缘情况（需要 LLM 深度分析）

def hybrid_approach(content: str):
    # 1. 先用 Workflow 快速检查（基于规则）
    workflow_result = workflow_app.invoke({"content": content})

    # 2. 如果 Workflow 决策明确 (approved 或 rejected)，直接返回
    if workflow_result["decision"] in ["approved", "rejected"]:
        return workflow_result

    # 3. 否则（needs_review），使用 Agent 做深度分析
    agent_result = agent_app.invoke({"content": content})
    return agent_result